# 08 Two checks on the 41 omitted providers
When I reviewed the draft I wanted two things settled before I send NHS England a question or publish a sentence about these providers.

1. Is the omission a steady feature of the year, or are some of the 41 providers appearing and disappearing? In particular, was NL7 (Assura Vertis), the one provider added when October was revised, a late arrival or a regular reporter?
2. Several of the 41 show exactly 100% within four hours. How much does that matter to the 0.7 point finding, and is it unique to the omitted providers?

I also classify each provider by the department type it mainly reports under, so that I describe the 41 from a field in the data and not from their names.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import numpy as np
import common as c
pd.options.display.float_format = '{:,.4f}'.format
pd.options.display.width = 200


In [2]:
m = pd.read_parquet(c.SILVER / 'msitae_monthly_provider.parquet')
cqi = pd.read_parquet(c.SILVER / 'cqi_long.parquet')
omitted = pd.read_parquet(c.GOLD / 'omitted_providers.parquet')
# Booked attendances are included, as the gate in notebook 03 showed they must be.
m['att'] = m[c.ATT_COLS].sum(axis=1) + m[c.BOOKED_COLS].sum(axis=1)
m['over4'] = m[c.OVER4_COLS].sum(axis=1) + m[c.OVER4_BOOKED_COLS].sum(axis=1)
codes = set(omitted.org_code)
a12 = cqi[(cqi.measure_id == 'AEQI012') & (cqi.org_code != 'ENG')]
inc = set(a12[a12.measure_value > 0].org_code)
print(len(codes), 'omitted providers;', len(inc), 'providers in CQI with report-table attendances')

41 omitted providers; 154 providers in CQI with report-table attendances


## 1. How many months does each omitted provider report in?

In [3]:
active = m[m.att > 0]
months_active = active[active.org_code.isin(codes)].groupby('org_code').agg(org_name=('org_name', 'first'), months_active=('month', 'nunique'), first_month=('month', 'min'), last_month=('month', 'max'), attendances=('att', 'sum'))
dist = months_active.months_active.value_counts().sort_index().rename_axis('months with activity').rename('providers').reset_index()
all_months = sorted(m.month.unique())
months_active['continuous_run'] = [(all_months.index(l) - all_months.index(f) + 1) == n for f, l, n in zip(months_active.first_month, months_active.last_month, months_active.months_active)]
part = months_active[months_active.months_active < 12]
print('of the', len(months_active), 'omitted providers,', int((months_active.months_active == 12).sum()), 'report in all twelve months and', len(part), 'report in fewer than twelve;', int(part.continuous_run.sum()), 'of those', len(part), 'report in one unbroken run and', int((~part.continuous_run).sum()), 'has a gap. I cannot tell from the data why.')
dist

of the 41 omitted providers, 30 report in all twelve months and 11 report in fewer than twelve; 10 of those 11 report in one unbroken run and 1 has a gap. I cannot tell from the data why.


,months with activity,providers
0,1,2
1,3,1
2,6,4
3,11,4
4,12,30


In [4]:
# The ones that do not report all year, so I can see who they are and when. I do not know why they report fewer months.
months_active[months_active.months_active < 12].sort_values(['months_active', 'attendances'], ascending=[True, False])

,org_name,months_active,first_month,last_month,attendances,continuous_run
org_code,,,,,,
NNJ10,ENDERBY LEISURE CENTRE,1,2026-03,2026-03,493,True
NNJ11,LUTTERWORTH HOSPITAL,1,2026-03,2026-03,299,True
Y07709,READING URGENT CARE CENTRE,3,2025-04,2025-06,5953,True
Y02147,URGENT CARE CENTRE (QMS),6,2025-04,2025-09,49604,True
A5D4Q,NEMS UTC,6,2025-04,2025-09,22491,True
NQT51,STATION PLAZA HEALTH CENTRE,6,2025-04,2025-09,6372,True
NVE43,HASTINGS GP LED HEALTH CENTRE,6,2025-10,2026-03,6028,True
AQM01,FAVERSHAM MEDICAL PRACTICE MIU,11,2025-05,2026-03,32511,True
NQT1I,CHIPPENHAM COMMUNITY HOSPITAL,11,2025-05,2026-03,18094,True


### NL7
The first-published October file had no NL7 row; the revised one added 3,565 attendances. I look at NL7 in every month, in the final files and in the first-published files I could retrieve.

In [5]:
prov, _ = c.read_msitae_monthly('msitae_provisional')
prov['att'] = prov[c.ATT_COLS + c.BOOKED_COLS].sum(axis=1)
months = sorted(m.month.unique())
nl7 = pd.DataFrame({'final': m[m.org_code == 'NL7'].set_index('month').att,
                    'first_published': prov[prov.org_code == 'NL7'].set_index('month').att}).reindex(months)
nl7['first_published_file_retrieved'] = nl7.index.isin(prov.month.unique())
n_cqi_rows = int((cqi.org_code == 'NL7').sum())
print('NL7 rows anywhere in CQI, any measure:', n_cqi_rows)
assert nl7.final.notna().all(), 'NL7 is missing from a final month'
assert n_cqi_rows == 0
nl7

NL7 rows anywhere in CQI, any measure: 0


,final,first_published,first_published_file_retrieved
month,,,
2025-04,3148,"3,148.0000",True
2025-05,3179,"3,179.0000",True
2025-06,2863,"2,863.0000",True
2025-07,2863,NaN,False
2025-08,3085,NaN,False
2025-09,3412,NaN,False
2025-10,3565,NaN,True
2025-11,3460,"3,460.0000",True
2025-12,3101,"3,101.0000",True


## 2. Which department type do they mainly report under?
I take each provider's largest attendance column (Type 1, Type 2 or Other A&E, booked attendances included) over the year. This is my classification from the sitrep columns; the data has no single type field.

In [6]:
yr = m[m.att > 0].groupby('org_code').agg(org_name=('org_name', 'first'), att=('att', 'sum'), over4=('over4', 'sum'),
                                          t1=('att_t1', 'sum'), bk1=('bkd_t1', 'sum'), t2=('att_t2', 'sum'), bk2=('bkd_t2', 'sum'),
                                          oth=('att_other', 'sum'), bko=('bkd_other', 'sum'), months=('month', 'nunique'))
yr['type1'] = yr.t1 + yr.bk1; yr['type2'] = yr.t2 + yr.bk2; yr['other'] = yr.oth + yr.bko
assert (yr.type1 + yr.type2 + yr.other == yr.att).all()
yr['in_cqi'] = yr.index.isin(inc)
yr['four_hour_pct'] = 100 * (1 - yr.over4 / yr.att)
yr['main_type'] = yr[['type1', 'type2', 'other']].idxmax(axis=1).map({'type1': 'Type 1', 'type2': 'Type 2', 'other': 'Other A&E'})
by_type = pd.crosstab(yr.in_cqi.map({True: 'in CQI', False: 'omitted from CQI'}), yr.main_type)
by_type

main_type,Other A&E,Type 1,Type 2
in_cqi,,,
in CQI,41,111,2
omitted from CQI,41,0,0


In [7]:
# The sharper statement behind the crosstab: how many of the omitted attendances sit in each department column?
g = m[m.org_code.isin(codes)]
cols = {'Type 1': ['att_t1', 'bkd_t1'], 'Type 2': ['att_t2', 'bkd_t2'], 'Other A&E': ['att_other', 'bkd_other']}
omit_by_col = pd.Series({k: int(g[v].sum().sum()) for k, v in cols.items()})
print(omit_by_col.to_string(), '| total', int(omit_by_col.sum()))
assert omit_by_col['Type 1'] == 0 and omit_by_col['Type 2'] == 0 and omit_by_col.sum() == int(omitted.att.sum())
# Two providers report booked attendances only (no non-booked rows). I list them because a classification that ignores booked attendances loses them.
y = m.groupby('org_code')[['att_t1', 'att_t2', 'att_other', 'bkd_t1', 'bkd_t2', 'bkd_other']].sum()
booked_only = y[(y[['att_t1', 'att_t2', 'att_other']].sum(axis=1) == 0) & (y[['bkd_t1', 'bkd_t2', 'bkd_other']].sum(axis=1) > 0)].index.tolist()
print('providers with booked attendances only:', booked_only, '| in CQI:', [o in inc for o in booked_only])

Type 1            0
Type 2            0
Other A&E    836365 | total 836365
providers with booked attendances only: ['8J094', 'Y03007'] | in CQI: [True, False]


In [8]:
# The Type 1 and Type 2 providers among the omitted, so I know which they are.
yr[(~yr.in_cqi) & (yr.main_type != 'Other A&E')][['org_name', 'att', 'main_type', 'four_hour_pct']]

,org_name,att,main_type,four_hour_pct
org_code,,,,


## 3. The providers at 100% within four hours

In [9]:
mm = m[m.org_code.isin(codes) & (m.att > 0)]
zero_every_month = mm.groupby('org_code').over4.apply(lambda s: bool((s == 0).all()))
z_om = yr[(~yr.in_cqi) & (yr.over4 == 0)]
z_in = yr[yr.in_cqi & (yr.over4 == 0)]
print('omitted providers with zero breaches in every month they report:', int(zero_every_month.sum()), 'of', len(zero_every_month))
print('omitted providers with zero breaches across the whole year:', len(z_om), '| attendances', int(z_om.att.sum()))
print('providers IN CQI with zero breaches across the whole year:', len(z_in), '| attendances', int(z_in.att.sum()))
print('smallest monthly attendance among the omitted zero-breach providers:', int(mm[mm.org_code.isin(zero_every_month[zero_every_month].index)].att.min()))
print()
other_in = yr[yr.in_cqi & (yr.main_type == 'Other A&E')].four_hour_pct
other_om = yr[(~yr.in_cqi) & (yr.main_type == 'Other A&E')].four_hour_pct
summary = pd.DataFrame({'providers': [len(other_in), len(other_om)], 'median four-hour %': [other_in.median(), other_om.median()],
                        'share at 99.9% or above': [(other_in >= 99.9).mean(), (other_om >= 99.9).mean()]},
                       index=['Other A&E providers in CQI', 'Other A&E providers omitted from CQI'])
summary

omitted providers with zero breaches in every month they report: 16 of 41
omitted providers with zero breaches across the whole year: 16 | attendances 272430
providers IN CQI with zero breaches across the whole year: 3 | attendances 101738
smallest monthly attendance among the omitted zero-breach providers: 52



,providers,median four-hour %,share at 99.9% or above
Other A&E providers in CQI,41,97.6872,0.0976
Other A&E providers omitted from CQI,41,99.9488,0.5610


## 4. Does it matter for the 0.7 point finding?
The zero-breach figure may reflect how those sites record times. So I replace their breach count with an assumed four-hour rate and watch the gap. The gap is CQI-providers-only minus all-providers, as in notebook 04.

In [10]:
cqi_att, cqi_over = m[m.org_code.isin(inc)].att.sum(), m[m.org_code.isin(inc)].over4.sum()
om_att, om_over = m[m.org_code.isin(codes)].att.sum(), m[m.org_code.isin(codes)].over4.sum()
zs = yr[(~yr.in_cqi) & (yr.over4 == 0)]
rows = []
for assumed in [None, 99, 98, 95, 90]:
    extra = 0 if assumed is None else zs.att.sum() * (1 - assumed / 100)
    om_over_adj = om_over + extra
    all_pct = 100 * (1 - (cqi_over + om_over_adj) / (cqi_att + om_att))
    cqi_pct = 100 * (1 - cqi_over / cqi_att)
    rows.append({'assumed four-hour % for the zero-breach omitted sites': 'as published' if assumed is None else assumed,
                 'omitted group four-hour %': 100 * (1 - om_over_adj / om_att), 'all providers %': all_pct,
                 'CQI providers only %': cqi_pct, 'gap (pp)': cqi_pct - all_pct})
sens = pd.DataFrame(rows)
sens

,assumed four-hour % for the zero-breach omitted sites,omitted group four-hour %,all providers %,CQI providers only %,gap (pp)
0,as published,97.8717,74.9288,74.2217,-0.7070
1,99,97.5460,74.9190,74.2217,-0.6973
2,98,97.2203,74.9093,74.2217,-0.6876
3,95,96.2431,74.8801,74.2217,-0.6583
4,90,94.6144,74.8314,74.2217,-0.6097


In [11]:
# The baseline row must reproduce notebook 04. Then I state what I expect: the gap stays beyond 0.6 points even at 90%.
assert abs(sens['gap (pp)'].iloc[0] - (-0.707)) < 0.001
assert (sens['gap (pp)'] < -0.6).all()
months_active.reset_index().to_parquet(c.GOLD / 'omitted_months_active.parquet', index=False)
nl7.reset_index().rename(columns={'index': 'month'}).to_parquet(c.GOLD / 'nl7_by_month.parquet', index=False)
by_type.reset_index().rename(columns={'in_cqi': 'coverage'}).to_parquet(c.GOLD / 'provider_main_type_by_coverage.parquet', index=False)
sens_out = sens.copy()
sens_out.columns = ['assumed_rate_for_zero_breach_sites', 'omitted_group_pct', 'all_providers_pct', 'cqi_providers_only_pct', 'gap_pp']
sens_out['assumed_rate_for_zero_breach_sites'] = sens_out.assumed_rate_for_zero_breach_sites.astype(str)
sens_out.to_parquet(c.GOLD / 'zero_breach_sensitivity.parquet', index=False)
print('The 0.7 point gap survives every assumption I tried.')

The 0.7 point gap survives every assumption I tried.
